# VectorNet H3 coverage maps

Interactive Folium maps of **H3 resolution-4** cells occupied by VectorNet detections in `data/02_interim/vectors_normalized.parquet`.

**What hexes mean:** a cell is drawn if VectorNet has ≥1 dated detection there (after optional filters). Empty space is *not* proven vector absence — only lack of records in this extract.

**Metrics are stored per `(h3_cell, vector_group)`:**
- `n_year_months` — how often detections appear (distinct year×month)
- `n_years` / `year_span` — how long the record spans
- `n_records` — raw detection row count (effort-sensitive)

**Map UX:** one polygon per H3 cell on an OpenStreetMap street basemap (hex fill is semi-transparent). Fill color uses the **max** of the plotted metric across groups in that cell (shared color scale).

**Hover or click** a hex for a **horizontal** table (metric rows × one column per vector group). When several groups are present, the **layer control** (top-right) toggles which groups appear as columns in that popup — it does not remove hexes from the map.

Set `VECTOR_GROUP` to filter the dataset to one group, or `None` for all groups.

After pulling code changes: **restart the kernel** and re-run from the top so `vector_guard.viz` reloads.

Edit the **config** cell (`COUNTRY_ISO2`, `VECTOR_GROUP`) and re-run from there to redraw.

In [ ]:
from pathlib import Path

import pandas as pd

from vector_guard.data import default_interim_dir, find_repo_root
from vector_guard.viz import aggregate_interim_vector_cells, folium_h3_choropleth

REPO_ROOT = find_repo_root(Path.cwd())
VECTOR_PATH = default_interim_dir(REPO_ROOT) / "vectors_normalized.parquet"
vectors = pd.read_parquet(VECTOR_PATH)
print(f"Loaded {len(vectors):,} rows from {VECTOR_PATH.relative_to(REPO_ROOT)}")
print("Countries:", vectors["country_iso2"].nunique())
print("Vector groups:", sorted(vectors["vector_group"].dropna().unique()))

## Config

Set `COUNTRY_ISO2` to an ISO-2 code (e.g. `"ES"`) to filter, or `None` for all countries.  
Set `VECTOR_GROUP` to `"mosquito"`, `"tick"`, `"biting_midge"`, or `"sandfly"` for a single group, or `None` to include all groups (multi-group cells list every group in the click popup).

In [ ]:
COUNTRY_ISO2 = None  # e.g. "ES"; None = all countries
VECTOR_GROUP = None  # e.g. "tick"; None = all groups

In [ ]:
filtered = vectors.copy()
if COUNTRY_ISO2 is not None:
    code = str(COUNTRY_ISO2).strip().upper()
    filtered = filtered.loc[filtered["country_iso2"].astype(str).str.upper() == code]
if VECTOR_GROUP is not None:
    group = str(VECTOR_GROUP).strip()
    filtered = filtered.loc[filtered["vector_group"].astype(str) == group]

cell_stats = aggregate_interim_vector_cells(
    vectors,
    country_iso2=COUNTRY_ISO2,
    vector_group=VECTOR_GROUP,
)

n_cells = int(cell_stats["h3_cell"].nunique()) if not cell_stats.empty else 0
n_multi = (
    int((cell_stats.groupby("h3_cell").size() >= 2).sum()) if not cell_stats.empty else 0
)

print(
    f"Filter: country={COUNTRY_ISO2!r}, vector_group={VECTOR_GROUP!r}"
)
print(f"Rows after filter: {len(filtered):,} / {len(vectors):,}")
print(f"(h3_cell, vector_group) units: {len(cell_stats):,}")
print(f"Unique H3 cells: {n_cells:,}")
print(f"H3 cells with ≥2 vector groups: {n_multi:,}")
if not cell_stats.empty:
    print("Units by vector_group:")
    display(cell_stats["vector_group"].value_counts())
    display(
        cell_stats[["n_year_months", "n_years", "year_span", "n_records"]].describe()
    )

## Frequency map (`n_year_months`)

Fill color = **max** `n_year_months` across vector groups in the cell. Hover/click a hex for a horizontal stats table (one column per vector group).

In [ ]:
freq_title = "n_year_months"
if COUNTRY_ISO2:
    freq_title += f" · {str(COUNTRY_ISO2).upper()}"
if VECTOR_GROUP:
    freq_title += f" · {VECTOR_GROUP}"

folium_h3_choropleth(cell_stats, "n_year_months", title=freq_title)

## Duration map (`n_years`)

Fill color = **max** `n_years` across vector groups in the cell. Hover/click a hex for a horizontal stats table (one column per vector group).

In [ ]:
dur_title = "n_years"
if COUNTRY_ISO2:
    dur_title += f" · {str(COUNTRY_ISO2).upper()}"
if VECTOR_GROUP:
    dur_title += f" · {VECTOR_GROUP}"

folium_h3_choropleth(cell_stats, "n_years", title=dur_title)

## Top cells

Highest frequency and longest duration under the current filter (one row per `h3_cell` × `vector_group`).

In [ ]:
cols = [
    "h3_cell",
    "vector_group",
    "n_year_months",
    "n_years",
    "year_span",
    "n_records",
    "year_min",
    "year_max",
]

print("Top by n_year_months")
display(cell_stats[cols].head(15))

print("Top by n_years")
display(
    cell_stats.sort_values(
        ["n_years", "year_span", "n_year_months"],
        ascending=False,
    )[cols].head(15)
)